## Paired Comparison Methods for Ranking Leaderboard Data

### This notebook makes use of the `choix` library of paired choice ranking and analysis functions

In [1]:
pip install choix numpy pandas jinja2


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

import pandas as pd
import json
import ast 

import numpy as np
import choix

### Motivation

In the generative AI ecosystem, leaderboards have become very popular for tracking the relative performance of LLMs and harnesses.
The purpose of creating a leaderboard is to assist people in making data driven choices from a variety of possible options.
However, there are some complexities with making use of leaderboard data. 

Consider the following scenario.
![Motivational Scenario](images/motivation.png)

As in the figure above, leaderboards often incorporate runs from multiple benchmarks.
The example above includes five benchmarks, A through E.
An advantage is that one can consider benchmark data from one or more use cases.
One disadvantage is that it introduces the problem of how to define a "global" ranking.
Taking the mean across benchmarks can provide an answer, but often the data do not include runs from all the same benchmarks.
You can see from the table above that no model has been run against the same set of benchmarks,
and so simply averaging scores must grapple with the fact that the average means something different for each model.
Does comparing means make sense when those means are taken over differing benchmark subsets?

One may also encounter situations where benchmark comparisons do not yield consistent ordering results.
In the figure above, benchmark B gives a different ordering of models X and Y than either benchmark A or C.
However, since X beats Y in 2 of 3 trials, we might wish to conclude that X beats Y, considering all the data we have available.

Lastly, we might wish to infer a conclusion about ordering of pairs which have no actual runs to compare them with.
For example, the above data do not provide any direct comparison between models X and Z,
but if we make an assumption of transitivity, we might wish to conclude that since model X is better than model Y,
and model Y is better than model Z, that model X is also better than model Z.
Bear in mind that this is an _assumption_, but it is a reasonable assumption based on the common property of transitivity.

One relevant conclusion from all of the above
is that it is best when we can pick one particular benchmark that represents our use case with good fidelity,
and simply use that benchmark to order our candidate models.
In some cases, however, we may not know _a priori_ what our use cases will be.
Or we might need to choose our models to cover a wide range of use cases.

In such situations, we may wish to identify a "best" model that takes into account all of the ambiguities described above.
For these situations, **paired comparison methods** are a good tool for imposing a single ordering
in the presence of these ambiguities, making use of the intuitive assumptions listed above. 

#### Let's run the example above with a paired choice ranking method from the `choix` library. We can see that it ranks the models according to the assumptions we described, in the presence of various ambiguous comparative runs:

In [3]:
# categories are the model names
cats = ["Model X", "Model Y", "Model Z"]

# (winner, loser) for each comparison
comps = [
    (0, 1),  # Model X beats Model Y  (benchmark A)
    (1, 0),  # Model Y beats Model X  (benchmark B)
    (0, 1),  # Model X beats Model Y  (benchmark C)
    (1, 2),  # Model Y beats Model Z  (benchmark D)
    (1, 2),  # Model Y beats Model Z  (benchmark E)
]

# run the pairwise comparison to get a ranking
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking = np.argsort(params, descending=True)

print("Ranking:")
for rank, idx in enumerate(ranking, start=1):
    print(f"  {rank:2d}. {params[idx]:+.3f}  {cats[idx]}")


Ranking:
   1. +2.627  Model X
   2. +1.938  Model Y
   3. -4.564  Model Z


#### The huggingface leaderboard I used for this notebook stores each benchmark run in its own JSON file. The following function reads all the run data in a directory and assembles it into a single pandas dataframe

In [4]:
def load_dicts_to_df(
    directory: str | Path,
    pattern: str = "*",
) -> pd.DataFrame:
    directory = Path(directory)
    if not directory.is_dir():
        raise NotADirectoryError(directory)

    rows: list[dict[str, Any]] = []
    # load any json files into the rows list
    for path in sorted(directory.glob(pattern)):
        if not path.is_file() or path.name.startswith("."):
            continue
        suffix = path.suffix.lower()
        data = None
        if suffix == ".json":
            with path.open(encoding="utf-8") as f:
                data = json.load(f)
        else:
            text = path.read_text(encoding="utf-8")
            data = ast.literal_eval(text)

        if not isinstance(data, dict):
            raise TypeError(f"{path} does not contain a dict (got {type(data).__name__})")
        rows.append(data)

    # the leaderboard json have a nested structure.
    # this function flattens the structure into flat column names
    return pd.json_normalize(rows)

#### Load the leaderboard data into a pandas dataframe

In [5]:
# git clone https://huggingface.co/spaces/taagarwa/coding-agent-leaderboard
df = load_dicts_to_df("/home/eje/git/coding-agent-leaderboard/results", pattern="*.json")
df.columns

Index(['benchmark.name', 'benchmark.repo', 'benchmark.num_tasks',
       'benchmark.url', 'harness.name', 'harness.skills', 'harness.is_oss',
       'harness.url', 'model.name', 'model.repo', 'model.is_oss',
       'model.num_params', 'model.precision', 'model.url', 'environment.name',
       'environment.config.path', 'environment.config.name',
       'environment.config.version', 'environment.config.ref',
       'environment.config.registry_url', 'environment.config.registry_path',
       'environment.config.overwrite', 'environment.config.download_dir',
       'environment.config.task_names',
       'environment.config.exclude_task_names', 'environment.config.n_tasks',
       'environment.url', 'metrics.n_tasks', 'metrics.n_errors',
       'metrics.score', 'metrics.n_input_tokens', 'metrics.n_cache_tokens',
       'metrics.n_output_tokens', 'metrics.n_total_tokens',
       'metrics.agent_time_seconds', 'metrics.total_time_seconds',
       'metrics.cost_usd', 'metrics.mean_input_toke

 #### The remainder of this notebook uses the following columns
 #### For the purposes of clarity, we will focus only on open models, and open harnesses, not proprietary

In [6]:
# Let's only consider open models
df = df.loc[df['model.is_oss'] == True]
# in fact let's only consider open harnesses too
df = df.loc[df['harness.is_oss'] == True]
df = df.reset_index(drop=True)
# keep only the columns we need
df = df[['model.name', 'harness.name', 'metrics.score', 'metrics.mean_cost_usd_per_task', 'metrics.mean_tokens_per_task', 'benchmark.name']]

#### Remove some dirty data so the rankings behave better

In [7]:
df = df.dropna()
df = df.loc[df['metrics.mean_tokens_per_task'] > 0]
df = df.reset_index(drop=True)

#### Our paired comparison method uses individual comparisons of the form (winner, loser). The following function can take a pandas table, and column names specifying which values are to be compared, and prepare the corresponding list of (winner, loser) pairs.

In [8]:
from typing import Any
from itertools import groupby
def prepare_ranking_data(df: pd.DataFrame,
                         catcol: str | list[str],
                         metcol: str,
                         descending: bool = False,
                         eqvcol: str | list[str] = []) -> tuple[list[tuple[int, int]], list[Any], list[float]]:
    """
    Prepare data for ranking comparisons.

    This function takes a pandas DataFrame and column names specifying which values
    are to be compared, and prepares the corresponding list of (winner, loser) pairs.

    Args:
        df: pandas DataFrame containing the data to be prepared for ranking
        catcol: column name or list of column names specifying the categories to be compared
        metcol: column name specifying the metric to be used for comparison
        descending: whether the metric is to be maximized (False) or minimized (True)
        eqvcol: column name or list of column names specifying equivalence groups.
                If specified, comparisons will only be made *within* each equivalence group.
    Returns:
        a tuple (comps, cats, eqvdata) where comps is a list of (winner, loser) pairs,
        and cats is a list of categories. The indices in the comps pairs
        correspond to the indices in the cats list.
        eqvdata[eqv][c][j] is the jth metric value for the cth category in equivalence group eqv.
    """
    ndata = df.shape[0]
    if ndata < 2:
        raise ValueError("Not enough data to prepare ranking comparisons")
    catcol = catcol if isinstance(catcol, list) else [catcol]
    eqvcol = eqvcol if isinstance(eqvcol, list) else [eqvcol]
    ncat = len(catcol)
    neqv = len(eqvcol)
    if ncat < 1:
        raise ValueError("No category column provided")
    tcols = catcol + eqvcol + [metcol]
    t = list(df[tcols].itertuples(index=False, name=None))
    metvals = [x[-1] for x in t]
    if ncat > 1:
        catvals = [x[:ncat] for x in t]
    else:
        # single category column: categories are just the values of the column
        catvals = [x[0] for x in t]
    if neqv > 1:
        # we will only compare items in the same equivalence group
        eqvvals = [x[ncat:ncat+neqv] for x in t]
    elif neqv == 1:
        # single equivalence column: equivalence groups are just the values of the column
        eqvvals = [x[ncat] for x in t]
    else:
        # by default all items are treated as one equivalence group
        eqvvals = ['[ALL]'] * ndata
    # map item categories to unique integers
    umap = dict([(y,x) for x,y in enumerate(sorted(set(catvals)))])
    cats = sorted(umap.keys())
    eqvcats = sorted(set(eqvvals))
    eqvdata = {}
    for eqv in eqvcats:
        # [[]] * len(cats) is bad because the lists are references to same list
        eqvdata[eqv] = [[] for _ in range(len(cats))]
    compvals = [(umap[c],m,e) for c,m,e in zip(catvals, metvals, eqvvals)]
    comps = []
    for i in range(ndata):
        ic, im, ie = compvals[i]
        for j in range(i):
            jc, jm, je = compvals[j]
            if ie != je:
                # ignore items with different equivalence values
                continue
            if im == jm:
                # ignore items with same metric value
                continue
            eqvdata[ie][ic].append(im)
            eqvdata[je][jc].append(jm)
            # pairs are always of form (winner, loser)
            iwin = im < jm if descending else im > jm
            if iwin:
                comps.append((ic, jc))
            else:
                comps.append((jc, ic))
    return comps, cats, eqvdata

### Combining rankings with metric information
One inconvenient property of paired-choice ranking is that it does not provide any perspective
in terms of whether two candidates might be close in performance or very different.
To get that perspective, we might wish to make use of the actual metric values.
However, as we discussed above, the metric data themselves have weaknesses.

The following utility function ranks the candidates using the paired choice ranking,
but also displays parallel columns for each equivalence class
(in the case of this study, the benchmark names are the equivalence class).

This allows a user to examine metric data (in our case, benchmark scores)
while being able to understand which options were run against which benchmark,
or whether some option combinations did not run against a benchmark.

In the case where multiple runs may have happend against a candiate category,
the mean is entered into the data table.

In [9]:
def ranking_dataframe(cats, params, eqvdata):
    ranking = np.argsort(params, descending=True)

    rankdata = {
        'rank': [], 'rank param': [], 'category': [],
    }
    for k in eqvdata.keys():
        rankdata[k] = []
    for rank, idx in enumerate(ranking, start=1):
        rankdata['rank'].append(rank)
        rankdata['rank param'].append(params[idx])
        rankdata['category'].append(cats[idx])
        for k in eqvdata.keys():
            mets = eqvdata[k][idx]
            mean = np.mean(mets) if len(mets) > 0 else pd.NA 
            rankdata[k].append(mean)
    df = pd.DataFrame(rankdata)
    style = {}
    style["rank param"] = "{:+5.2f}"
    for k in eqvdata.keys():
        style[k] = "{:.3f}"
    # can only call style.format once
    df = df.style.format(style)
    df = df.set_properties(**{'font-family': 'monospace'})
    return df

#### Rank our harnesses, using benchmark score as our metric. We will "integrate" over models, and only using comparisons where the benchmark is the same:

In [10]:
comps, cats, eqvdata = prepare_ranking_data(df, 'harness.name', 'metrics.score', eqvcol='benchmark.name')
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking_dataframe(cats, params, eqvdata)

,rank,rank param,category,RH SWE-Bench,SWE-Bench Pro -- Ansible,SWE-Bench Verified,Terminal Bench 2.0
0,1,+0.59,Qwen Code,,0.438,0.638,
1,2,-0.24,Pi,0.305,0.408,0.480,0.360
2,3,-0.35,OpenCode,0.347,0.402,0.475,0.303


#### Rank our models using benchmark score, integrating over harness, and comparing within benchmark:

In [11]:
comps, cats, eqvdata = prepare_ranking_data(df, 'model.name', 'metrics.score', eqvcol='benchmark.name')
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking_dataframe(cats, params, eqvdata)

,rank,rank param,category,RH SWE-Bench,SWE-Bench Pro -- Ansible,SWE-Bench Verified,Terminal Bench 2.0
0,1,+4.55,Qwen3.6-27B-FP8,0.454,0.531,0.668,
1,2,+1.71,Qwen3.6-35B-A3B-NVFP4,,0.432,0.612,0.332
2,3,+1.42,Gemma4-31B-FP8,,0.443,0.590,
3,4,-2.10,Nemotron-3-Super-120B-NVFP4,0.262,0.348,0.454,
4,5,-2.50,Mistral-Small-4-119B-2603-NVFP4,,0.360,0.275,
5,6,-3.07,GPT-OSS-120B,0.262,0.312,0.280,


#### Rank (model, harness) pairs by benchmark score, comparing only within benchmark

In [12]:
comps, cats, eqvdata = prepare_ranking_data(df, ['model.name', 'harness.name'], 'metrics.score', eqvcol='benchmark.name')
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking_dataframe(cats, params, eqvdata)

,rank,rank param,category,RH SWE-Bench,SWE-Bench Pro -- Ansible,SWE-Bench Verified,Terminal Bench 2.0
0,1,+8.79,"('Qwen3.6-27B-FP8', 'Pi')",0.468,0.490,0.694,
1,2,+7.70,"('Qwen3.6-27B-FP8', 'OpenCode')",0.440,0.573,0.642,
2,3,+7.16,"('Qwen3.6-35B-A3B-NVFP4', 'Pi')",,0.479,0.650,0.360
3,4,+3.30,"('Qwen3.6-35B-A3B-NVFP4', 'Qwen Code')",,0.438,0.638,
4,5,+2.59,"('Gemma4-31B-FP8', 'Pi')",,0.469,0.574,
5,6,+1.88,"('Gemma4-31B-FP8', 'OpenCode')",,0.417,0.606,
6,7,-2.29,"('Qwen3.6-35B-A3B-NVFP4', 'OpenCode')",,0.375,0.548,0.303
7,8,-3.69,"('Mistral-Small-4-119B-2603-NVFP4', 'OpenCode')",,0.385,0.318,
8,9,-4.06,"('Nemotron-3-Super-120B-NVFP4', 'Pi')",0.216,0.375,0.498,
9,10,-4.39,"('Nemotron-3-Super-120B-NVFP4', 'OpenCode')",0.308,0.323,0.410,


#### Rank (model, harness) pairs, by expected dollar cost per run. Here, we specify that lower cost is better:

In [13]:
comps, cats, eqvdata = prepare_ranking_data(df, ['model.name', 'harness.name'], 'metrics.mean_cost_usd_per_task', eqvcol='benchmark.name', descending=True)
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking_dataframe(cats, params, eqvdata)

,rank,rank param,category,RH SWE-Bench,SWE-Bench Pro -- Ansible,SWE-Bench Verified,Terminal Bench 2.0
0,1,+9.70,"('GPT-OSS-120B', 'OpenCode')",0.010,0.010,0.010,
1,2,+6.40,"('Mistral-Small-4-119B-2603-NVFP4', 'Pi')",,0.040,0.020,
2,3,+6.37,"('GPT-OSS-120B', 'Pi')",0.010,0.030,0.030,
3,4,+3.19,"('Mistral-Small-4-119B-2603-NVFP4', 'OpenCode')",,0.070,0.030,
4,5,-0.65,"('Qwen3.6-35B-A3B-NVFP4', 'Qwen Code')",,0.100,0.070,
5,6,-0.95,"('Qwen3.6-35B-A3B-NVFP4', 'OpenCode')",,0.130,0.060,0.130
6,7,-1.69,"('Nemotron-3-Super-120B-NVFP4', 'OpenCode')",0.100,0.200,0.040,
7,8,-2.33,"('Qwen3.6-35B-A3B-NVFP4', 'Pi')",,0.140,0.080,0.120
8,9,-2.85,"('Nemotron-3-Super-120B-NVFP4', 'Pi')",0.090,0.270,0.070,
9,10,-3.06,"('Gemma4-31B-FP8', 'Pi')",,0.170,0.080,


#### Rank (model, harness) pairs by expected token cost: here lower cost is better:

In [14]:
comps, cats, eqvdata = prepare_ranking_data(df, ['model.name', 'harness.name'], 'metrics.mean_tokens_per_task', eqvcol='benchmark.name', descending=True)
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking_dataframe(cats, params, eqvdata)

,rank,rank param,category,RH SWE-Bench,SWE-Bench Pro -- Ansible,SWE-Bench Verified,Terminal Bench 2.0
0,1,+4.17,"('Mistral-Small-4-119B-2603-NVFP4', 'Pi')",,1105100.000,395947.000,
1,2,+3.82,"('Gemma4-31B-FP8', 'Pi')",,833841.000,763314.000,
2,3,+3.65,"('GPT-OSS-120B', 'Pi')",1179936.000,1310886.000,182024.000,
3,4,+3.13,"('GPT-OSS-120B', 'OpenCode')",1181932.000,1175236.000,408602.000,
4,5,+2.59,"('Gemma4-31B-FP8', 'OpenCode')",,1055516.000,1059493.000,
5,6,+2.59,"('Mistral-Small-4-119B-2603-NVFP4', 'OpenCode')",,1474932.000,755531.000,
6,7,+1.57,"('Qwen3.6-27B-FP8', 'OpenCode')",1156452.000,3609525.000,797177.000,
7,8,+0.93,"('Qwen3.6-35B-A3B-NVFP4', 'OpenCode')",,2174618.000,949488.000,553538.000
8,9,+0.07,"('Qwen3.6-35B-A3B-NVFP4', 'Qwen Code')",,1668444.000,1227106.000,
9,10,-1.16,"('Qwen3.6-27B-FP8', 'Pi')",1580647.000,4139532.000,1098072.000,
